# SynBioHub

![SynBioHub](https://proto-bio.github.io/proto-assets/images/tool/synbiohub/hero.png)

This notebook demonstrates `run_synbiohub_fetch`, which downloads one design from a SynBioHub registry as SBOL and returns it as typed parts and features. The SBOL is kept verbatim, and the output exports to SBOL, FASTA, or GenBank. The running example is the iGEM device `BBa_I13504`, a GFP expression cassette built from an RBS, the GFP coding sequence, and a double terminator.

The public SynBioHub instance requires a login for downloads, including public records. Set `SYNBIOHUB_TOKEN` before running; the Usage Tips in the toolkit README give the commands to create one.

In [1]:
from proto_tools.utils.notebook_docs import display_overview, display_api_reference, display_docs_section, display_doc_link, display_available_tools
display_doc_link("synbiohub")
display_overview("synbiohub")
display_docs_section("synbiohub", "Background")

# SynBioHub

The `synbiohub-fetch` tool retrieves one design from a caller-selected SynBioHub registry and returns it as typed parts and features, with the source SBOL kept verbatim. It maintains no local parts catalog.

[SynBioHub](https://synbiohub.org/) is a standards-enabled repository for synthetic biology designs. SBOL preserves components, nested subcomponents, sequence constraints, annotations, and provenance, allowing a circuit or construct to remain a composition of segments rather than only a flattened nucleotide string. The tool uses the documented SBOL download endpoint and authenticates with a SynBioHub login token.

## Available tools

In [2]:
display_available_tools("synbiohub")

- **`run_synbiohub_fetch()`** — Fetch a SynBioHub design as typed parts and features, exportable to SBOL, FASTA, and GenBank

### `run_synbiohub_fetch`

`run_synbiohub_fetch` takes the URI of one ComponentDefinition, downloads its recursive SBOL, and parses it. `design` is the requested part and `subparts` holds every part it references. The object must be on the same host as `registry_url`, so a URI cannot redirect retrieval to another server. The token is read from `SYNBIOHUB_TOKEN` and never appears in cache keys or outputs.

In [3]:
import os

from proto_tools import (
    SynBioHubFetchConfig,
    SynBioHubFetchInput,
    run_synbiohub_fetch,
)

assert os.environ.get("SYNBIOHUB_TOKEN"), "set SYNBIOHUB_TOKEN before running this notebook"

In [4]:
# Display docs
display_api_reference("synbiohub", "input", "run_synbiohub_fetch")

# One design on the public registry
inputs = SynBioHubFetchInput(object_uri="https://api.synbiohub.org/public/igem/BBa_I13504/1")

**Input** — `SynBioHubFetchInput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>object_uri</code> | <code>str</code> | required | Absolute URI of a ComponentDefinition on the selected SynBioHub registry |

In [5]:
# Display config docs
display_api_reference("synbiohub", "config", "run_synbiohub_fetch")

# Defaults use the public registry and read SYNBIOHUB_TOKEN | see docs above for all fields
config = SynBioHubFetchConfig()

**Config** — `SynBioHubFetchConfig`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>registry_url</code> | <code>str</code> | <code>'https://api.synbiohub.org'</code> | SynBioHub registry origin allowed for object retrieval |
| <code>synbiohub_token</code> | <code>str &#124; None</code> | <code>$SYNBIOHUB_TOKEN</code> | SynBioHub login token; required by the public instance. Falls back to $SYNBIOHUB_TOKEN. |

In [6]:
# Run the fetch
result = run_synbiohub_fetch(inputs, config)

In [7]:
# Display output docs
display_api_reference("synbiohub", "output", "run_synbiohub_fetch")

design = result.design
print(f"{design.display_id}: {design.description} ({design.molecule}, {len(design.sequence)} nt)")
print(f"Roles: {design.roles}")
print(f"{len(design.features)} features, {len(result.subparts)} subparts, {len(result.sbol):,} chars of SBOL")

**Output** — `SynBioHubFetchOutput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>object_uri</code> | <code>str</code> | required | Requested object URI |
| <code>registry_url</code> | <code>str</code> | required | Registry origin used for retrieval |
| <code>source_url</code> | <code>str</code> | required | SBOL download endpoint |
| <code>design</code> | <code>SynBioHubPart</code> | required | The requested design |
| <code>subparts</code> | <code>list[SynBioHubPart]</code> | <code>[]</code> | Every other part in the SBOL document, at any depth |
| <code>sbol</code> | <code>str</code> | required | The downloaded recursive SBOL document, verbatim |

**`SynBioHubPart`**

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>uri</code> | <code>str</code> | required | SBOL identity of the part |
| <code>display_id</code> | <code>str</code> | required | SBOL displayId |
| <code>name</code> | <code>str &#124; None</code> | <code>None</code> | Part title |
| <code>description</code> | <code>str &#124; None</code> | <code>None</code> | Part description |
| <code>molecule</code> | <code>Literal['dna', 'rna', 'protein', 'other']</code> | required | Molecule type of the part |
| <code>topology</code> | <code>Optional[Literal['linear', 'circular']]</code> | <code>None</code> | Linear or circular, when the registry records it |
| <code>roles</code> | <code>list[str]</code> | <code>[]</code> | Sequence Ontology terms such as SO:0000141 |
| <code>sequence</code> | <code>str &#124; None</code> | <code>None</code> | Part sequence, uppercase |
| <code>features</code> | <code>list[SynBioHubFeature]</code> | <code>[]</code> | Features annotated on the sequence; annotations without a range are omitted |
| <code>modified</code> | <code>datetime &#124; None</code> | <code>None</code> | Last modification time (dcterms:modified), else creation time |

**`SynBioHubFeature`**

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>display_id</code> | <code>str</code> | required | SBOL displayId of the sequence annotation |
| <code>name</code> | <code>str &#124; None</code> | <code>None</code> | Annotation title, else its subpart's title |
| <code>roles</code> | <code>list[str]</code> | <code>[]</code> | Sequence Ontology terms such as SO:0000139, inherited from the subpart if unset |
| <code>locations</code> | <code>list[SynBioHubLocation]</code> | required | Ranges the feature covers; more than one is a split feature |
| <code>subpart_uri</code> | <code>str &#124; None</code> | <code>None</code> | URI of the subpart this feature places, matching an entry in subparts |

**`SynBioHubLocation`**

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>start</code> | <code>int</code> | required | First position, 1-indexed and inclusive |
| <code>end</code> | <code>int</code> | required | Last position, 1-indexed and inclusive |
| <code>strand</code> | <code>Literal['+', '-']</code> | required | Strand relative to the part's sequence |

BBa_I13504: Screening plasmid intermediate (dna, 875 nt)
Roles: ['SO:0000804']
4 features, 4 subparts, 38,793 chars of SBOL


#### Read the Composition

Each feature places a subpart on the design sequence with 1-indexed, inclusive coordinates. `subpart_uri` points into `subparts`, which carries each part's own description, roles, and sequence.

In [8]:
subparts = {part.uri: part for part in result.subparts}

for feature in sorted(design.features, key=lambda f: f.locations[0].start):
    location = feature.locations[0]
    part = subparts[feature.subpart_uri]
    print(f"{location.start:>4}-{location.end:<4} ({location.strand})  {part.display_id:<10} {feature.roles}  {part.description}")

   1-12   (+)  BBa_B0034  ['SO:0000139']  RBS (Elowitz 1999) -- defines RBS efficiency
  19-738  (+)  BBa_E0040  ['SO:0000316']  green fluorescent protein derived from jellyfish Aequeora victoria wild-type GFP (SwissProt: P42212
 747-826  (+)  BBa_B0010  ['SO:0000141']  T1 from E. coli rrnB
 835-875  (+)  BBa_B0012  ['SO:0000141']  TE from coliphageT7


## Export Results

The output exports to JSON (the default), the verbatim SBOL, FASTA, or GenBank. FASTA and GenBank are views of the requested design; GenBank feature keys come from the Sequence Ontology roles.

In [9]:
from pathlib import Path

output_dir = Path("./example_output")
output_dir.mkdir(exist_ok=True)

for file_format, suffix in [("json", "json"), ("sbol", "sbol"), ("fasta", "fasta"), ("genbank", "gb")]:
    result.export("synbiohub_bba_i13504", export_path=output_dir, file_format=file_format)
    print(f"Exported to {output_dir / f'synbiohub_bba_i13504.{suffix}'}")

print()
print((output_dir / "synbiohub_bba_i13504.gb").read_text().split("ORIGIN")[0])

Exported to example_output/synbiohub_bba_i13504.json
Exported to example_output/synbiohub_bba_i13504.sbol
Exported to example_output/synbiohub_bba_i13504.fasta
Exported to example_output/synbiohub_bba_i13504.gb

LOCUS       BBa_I13504               875 bp    DNA              UNK 31-AUG-2015
DEFINITION  Screening plasmid intermediate.
ACCESSION   BBa_I13504
VERSION     BBa_I13504
KEYWORDS    .
SOURCE      .
  ORGANISM  .
            .
FEATURES             Location/Qualifiers
     RBS             1..12
                     /label="BBa_B0034"
     CDS             19..738
                     /label="BBa_E0040"
     terminator      747..826
                     /label="BBa_B0010"
     terminator      835..875
                     /label="BBa_B0012"

